# Chapter 2 &mdash; Manual Artifact Rejection & Bad Channels Rejection
## 2.1. Rationale
### 2.1.1. Manual Artifact Rejection

Manual artifact rejection (MAR) is the act of visually inspecting EEG data and flagging epochs for rejection with a mouse click. 

MAR is the simplest form of data cleaning, as it merely thrashes unusable data. It has mainly three use cases:

- Remove the largest artifacts before feeding the data to automatic procedures like ICA
    - This can increase the effectiveness of automatic procedures, for example yielding clearer ICA decompositions

- Remove leftover artifacts after automatic procedures
    - For example, one may remove ocular artifacts by rejecting ICA's ocular components, but that will not remove the artifacts due to occasional muscle contractions

- Remove the most obvious artifacts, and do no more
    - This is what purists do: no automatic procedures, only MAR (plus some filtering and bad channels rejection at most)

In all three cases, the criteria to follow for MAR are usually the following:

- The data are characterised by artifacts that present as high-frequency, high-amplitude fluctuations that cannot plausibly be cerebral
- The artifacts in question are circumscribed in time
- The artifacts in question affect multiple channels

The amount of epochs that can be rejected depends on the amount of epochs at hand: MAR must hit a trade-off between cleaning the data as much as possible and preserving as many epochs as possible to avoid losing statistical power. There is no universally valid number, but as a rule of thumb it would be better to keep around 100 epochs per condition if the goal is to analyse event-related potentials. 

### 2.1.2. Bad Channels Rejection
Bad channels rejection (BCR) is the act of visually inspecting data and flagging _whole channels_ for rejection.

BCR has the same use cases as MAR but it follows slightly different criteria:

- The data must be characterised by high-frequency, high-amplitude fluctuations that cannot be cerebral
- The artifacts in question must _not_ be circumscribed in time
- The artifacts in question must affect _a single_ channel

After rejection, bad channels can be reconstructed by neare-neighbour interpolation &mdash; that is, if channels $A$, $B$, and $C$ are close to one another, channel $B$ can be reconstructed as a function $A$ and $C$ after rejection.

## 2.2. Execution

To do MAR in MNE-Python, it is sufficient to plot the epoched data and click on the artifactual epochs once. Clicked epochs will turn red and that will mean that they were added to a "bads" list. Epochs added to the "bads" list will not count for subsequent operations, meaning that they will not be fed to any downstream preprocessing step nor taken into account in the computation of evoked potentials. However, they will not be deleted: epochs can go in and out of the bads list as many times as a user likes. To remove an epoch from the bads list, it is sufficient to click on it a second time.

To actually delete epochs from Python memory, MNE-Python provides a function called `drop_bad()`. Instead of adding epochs to a "bads" list, `drop_bad()` will actually delete them from Python memory (but not from your _computer's_ memory). This can be inconvenient because if you change your mind about one epoch, you cannot go back with ease: you need to recreate the variable at the state where it last had all the epochs, which might entail re-running one or more previous steps (unless you save intermediate preprocessing outputs, which you should).

All the above applies equally to BCR, with the exception that users should click on a channel's _label_ (and not on the actual voltage trace) to flag it for rejection.

The code below plots `post_filtering_epochs`, allowing users to scroll through them and flag them for rejection if necessary. In the plot, the red epochs affected by large artifacts are flagged for rejection and will be added to the "bads" list upon closing the plot.

In [ ]:
post_filtering_epochs.plot();

<p align="center">
<img src="files/mar.png" width="1000"/>
</p>

After rejecting one or more epochs, you can verify that they have been excluded from the data of interest by printing out a summary table. Here, the `Total number of events` field from the `Acquisition` section will decrease from the initial value (which, in the reference dataset, is 200) to a value that mirrors the number of rejected epochs. For example, rejecting 12 epochs in the reference dataset would change the events count from 200 to 188. Similarly, rejecting channels will change the `EEG` field from the `Channels` section from a number like `70` to a short sentence like `69 and 1 bad`.

The code below renames the epochs object to more closely mirror the post-rejection situation, then it prints out a summary table.

In [ ]:
post_rejection_epochs = post_filtering_epochs
post_rejection_epochs

Finally, the code below writes post_rejection epochs to file, then it computs their average to visualize the TEP.

In [ ]:
post_rejection_epochs.save(fname="data/post_rejection_epo.fif", 
                           overwrite=True)
post_rejection_tep = post_rejection_epochs.average()
post_rejection_tep.plot();